# Retail Sales Trend Analysis

## Project Overview
This notebook analyzes retail sales performance over time using the Sample Superstore dataset already available in this workspace. The focus is practical: identify trend direction, seasonality, strong and weak periods, leading categories, and operational signals that a retail team could actually use.

## Learning Objectives
- Build a time-based retail sales analysis workflow from raw transactional data.
- Separate overall revenue growth from seasonality and short-term noise.
- Compare category and sub-category performance using revenue, profit, and quantity together.
- Inspect discount and shipping-related patterns as potential revenue and profitability drivers.
- Translate descriptive analysis into cautious, actionable business recommendations.

## Business Problem
Retail leaders need to understand when sales are strongest, when performance softens, and which product lines drive growth versus margin risk. Without that baseline, inventory, promotion, and merchandising decisions stay reactive.

## Why This Analysis Matters
Trend and seasonality analysis helps teams plan staffing, campaign timing, product emphasis, and discount strategy. Category analysis helps distinguish between revenue generators and margin contributors, which is essential for better pricing and assortment decisions.

## Dataset Overview
The notebook uses the repo-local `Sample - Superstore.xls` workbook. The verified schema includes retail order fields such as `Order Date`, `Ship Date`, `Category`, `Sub-Category`, `Sales`, `Quantity`, `Discount`, and `Profit`, which makes it a strong fit for time-based and category-based retail analysis.

## Dataset Source And License Notes
This workbook is a widely circulated Sample Superstore training dataset and is already present in this repository. Because this dataset appears in many mirrors, confirm the license and redistribution terms of the specific source copy you use outside this workspace. This notebook only analyzes the local workbook already available here.

## Environment Setup
The next cell installs only the packages required to run this notebook from a clean Python environment. The workbook is in `.xls` format, so `xlrd` is required for reading it with pandas.

In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "xlrd": "xlrd",
}

def ensure_package(package_name, import_name=None):
    module_name = import_name or package_name
    try:
        importlib.import_module(module_name)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for package_name, import_name in REQUIRED_PACKAGES.items():
    ensure_package(package_name, import_name)

print("Environment setup complete.")

## Imports
These imports cover path handling, data validation, aggregation, and plotting. The notebook keeps dependencies minimal so the analysis stays easy to rerun.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
np.random.seed(42)

## Configuration / Constants
This block locates the workspace root, points to the workbook path, and defines the schema the notebook expects. Keeping these assumptions in one cell makes the analysis easier to audit later.

In [ ]:
def locate_workspace_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "README.md").exists():
            return candidate
    return start_path

WORKSPACE_ROOT = locate_workspace_root(Path.cwd())
DATASET_PATH = WORKSPACE_ROOT / "Time Series Analysis" / "Time Series Forecasting" / "Sample - Superstore.xls"
EXPECTED_COLUMNS = [
    "Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID",
    "Customer Name", "Segment", "Country", "City", "State", "Postal Code",
    "Region", "Product ID", "Category", "Sub-Category", "Product Name",
    "Sales", "Quantity", "Discount", "Profit"
]

config_preview = {
    "workspace_root": str(WORKSPACE_ROOT),
    "dataset_path": str(DATASET_PATH),
    "expected_column_count": len(EXPECTED_COLUMNS),
}
print(json.dumps(config_preview, indent=2))

## Dataset Loading
The dataset is loaded from a repo-local workbook. If the file is missing, the notebook raises a clear error with the expected path instead of failing later in less obvious ways.

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Expected workbook not found at: {DATASET_PATH}")

raw_df = pd.read_excel(DATASET_PATH)
print(f"Raw shape: {raw_df.shape}")
display(raw_df.head())

## Data Validation Checks
Before interpreting trends, this block verifies the schema, checks date parsing, reviews nulls, and looks for duplicate rows. This is the lowest-cost way to avoid drawing conclusions from broken inputs.

In [ ]:
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(raw_df.columns))
extra_columns = sorted(set(raw_df.columns) - set(EXPECTED_COLUMNS))

validation_df = raw_df.copy()
validation_df["Order Date"] = pd.to_datetime(validation_df["Order Date"], errors="coerce")
validation_df["Ship Date"] = pd.to_datetime(validation_df["Ship Date"], errors="coerce")

validation_report = pd.Series({
    "row_count": int(len(raw_df)),
    "missing_columns": missing_columns,
    "extra_columns": extra_columns,
    "invalid_order_dates": int(validation_df["Order Date"].isna().sum()),
    "invalid_ship_dates": int(validation_df["Ship Date"].isna().sum()),
    "duplicate_full_rows": int(raw_df.duplicated().sum()),
    "duplicate_order_product_rows": int(raw_df.duplicated(subset=["Order ID", "Product ID"]).sum()),
    "unique_orders": int(raw_df["Order ID"].nunique()),
    "unique_products": int(raw_df["Product ID"].nunique()),
    "unique_categories": int(raw_df["Category"].nunique()),
    "unique_subcategories": int(raw_df["Sub-Category"].nunique()),
}).to_frame(name="value")

missing_summary = raw_df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")

display(validation_report)
display(missing_summary)

## Data Cleaning And Feature Preparation
This cleaning step keeps the analysis conservative. It removes duplicate rows, parses dates explicitly, standardizes key string columns, and adds time-based fields needed for trend, seasonality, and moving-average analysis.

In [ ]:
df = raw_df.copy()
df = df.drop_duplicates().copy()
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")
df = df.dropna(subset=["Order Date", "Category", "Sub-Category", "Sales", "Profit", "Quantity"]).copy()

string_columns = ["Category", "Sub-Category", "Segment", "Region", "Ship Mode", "State", "City", "Product Name"]
for column in string_columns:
    df[column] = df[column].astype(str).str.strip()

df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
df["Profit"] = pd.to_numeric(df["Profit"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Discount"] = pd.to_numeric(df["Discount"], errors="coerce")
df = df.dropna(subset=["Sales", "Profit", "Quantity", "Discount"]).copy()

df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.month
df["Order Month Name"] = df["Order Date"].dt.month_name()
df["Order Quarter"] = df["Order Date"].dt.to_period("Q").astype(str)
df["Order Weekday"] = df["Order Date"].dt.day_name()
df["YearMonth"] = df["Order Date"].dt.to_period("M").astype(str)
df["Ship Delay Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Profit Margin Pct"] = np.where(df["Sales"] != 0, (df["Profit"] / df["Sales"]) * 100, np.nan)
df["Discount Bucket"] = pd.cut(df["Discount"], bins=[-0.001, 0, 0.1, 0.2, 0.4, 1.0], labels=["0%", "0-10%", "10-20%", "20-40%", "40%+"])

cleaning_report = pd.Series({
    "clean_rows": int(len(df)),
    "date_start": str(df["Order Date"].min().date()),
    "date_end": str(df["Order Date"].max().date()),
    "total_sales": float(df["Sales"].sum()),
    "total_profit": float(df["Profit"].sum()),
    "average_order_line_sales": float(df["Sales"].mean()),
}).to_frame(name="value")

display(cleaning_report)
display(df.head())

## Time-Based EDA
This section aggregates sales and profit over time so we can see long-run direction without getting lost in row-level transaction noise. It also helps separate overall growth from temporary spikes and dips.

In [ ]:
daily_trend = df.groupby("Order Date", as_index=False).agg(
    daily_sales=("Sales", "sum"),
    daily_profit=("Profit", "sum"),
    order_lines=("Order ID", "size"),
)
daily_trend["sales_ma_30"] = daily_trend["daily_sales"].rolling(30, min_periods=7).mean()
daily_trend["sales_ma_90"] = daily_trend["daily_sales"].rolling(90, min_periods=30).mean()

monthly_trend = df.groupby("YearMonth", as_index=False).agg(
    monthly_sales=("Sales", "sum"),
    monthly_profit=("Profit", "sum"),
    monthly_quantity=("Quantity", "sum"),
    monthly_orders=("Order ID", "nunique"),
)
monthly_trend["YearMonthDate"] = pd.to_datetime(monthly_trend["YearMonth"] + "-01")

fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=False)
axes[0].plot(daily_trend["Order Date"], daily_trend["daily_sales"], color="lightsteelblue", label="Daily sales")
axes[0].plot(daily_trend["Order Date"], daily_trend["sales_ma_30"], color="navy", linewidth=2, label="30-day moving average")
axes[0].plot(daily_trend["Order Date"], daily_trend["sales_ma_90"], color="darkorange", linewidth=2, label="90-day moving average")
axes[0].set_title("Daily Sales With Moving Averages")
axes[0].legend()

axes[1].plot(monthly_trend["YearMonthDate"], monthly_trend["monthly_sales"], marker="o", color="teal")
axes[1].set_title("Monthly Sales Trend")
axes[1].set_ylabel("Sales")

axes[2].plot(monthly_trend["YearMonthDate"], monthly_trend["monthly_profit"], marker="o", color="firebrick")
axes[2].set_title("Monthly Profit Trend")
axes[2].set_ylabel("Profit")

plt.tight_layout()
plt.show()

display(monthly_trend.head())

## Seasonality And Weak Periods
Trend alone is not enough. This block checks whether certain months or weekdays consistently outperform others and identifies weak periods by both sales and profit so revenue strength is not confused with healthy margin behavior.

In [ ]:
month_order = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

monthly_seasonality = df.groupby("Order Month Name", as_index=False).agg(
    average_sales=("Sales", "mean"),
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
)
monthly_seasonality["Order Month Name"] = pd.Categorical(monthly_seasonality["Order Month Name"], categories=month_order, ordered=True)
monthly_seasonality = monthly_seasonality.sort_values("Order Month Name")

weekday_seasonality = df.groupby("Order Weekday", as_index=False).agg(
    average_sales=("Sales", "mean"),
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
)
weekday_seasonality["Order Weekday"] = pd.Categorical(weekday_seasonality["Order Weekday"], categories=weekday_order, ordered=True)
weekday_seasonality = weekday_seasonality.sort_values("Order Weekday")

year_month_sales = df.groupby(["Order Year", "Order Month Name"], as_index=False)["Sales"].sum()
year_month_sales["Order Month Name"] = pd.Categorical(year_month_sales["Order Month Name"], categories=month_order, ordered=True)
year_month_pivot = year_month_sales.pivot(index="Order Year", columns="Order Month Name", values="Sales").reindex(columns=month_order)

weak_months = monthly_trend.nsmallest(5, "monthly_sales")[["YearMonth", "monthly_sales", "monthly_profit"]]
weak_profit_months = monthly_trend.nsmallest(5, "monthly_profit")[["YearMonth", "monthly_sales", "monthly_profit"]]

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=monthly_seasonality, x="Order Month Name", y="total_sales", ax=axes[0], palette="viridis")
axes[0].set_title("Total Sales By Month Name")
axes[0].tick_params(axis="x", rotation=45)

sns.barplot(data=weekday_seasonality, x="Order Weekday", y="total_sales", ax=axes[1], palette="magma")
axes[1].set_title("Total Sales By Weekday")
axes[1].tick_params(axis="x", rotation=45)

sns.heatmap(year_month_pivot, cmap="YlGnBu", ax=axes[2])
axes[2].set_title("Monthly Sales Heatmap By Year")

plt.tight_layout()
plt.show()

display(monthly_seasonality)
display(weekday_seasonality)
display(weak_months)
display(weak_profit_months)

## Category Analysis
This section compares category and sub-category performance using more than revenue alone. Looking at sales, profit, quantity, and average discount together helps separate headline winners from margin-risk product lines.

In [ ]:
category_summary = df.groupby("Category", as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    total_quantity=("Quantity", "sum"),
    average_discount=("Discount", "mean"),
)
category_summary["profit_margin_pct"] = np.where(category_summary["total_sales"] != 0, (category_summary["total_profit"] / category_summary["total_sales"]) * 100, np.nan)
category_summary = category_summary.sort_values("total_sales", ascending=False)

subcategory_summary = df.groupby(["Category", "Sub-Category"], as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    total_quantity=("Quantity", "sum"),
    average_discount=("Discount", "mean"),
)
subcategory_summary["profit_margin_pct"] = np.where(subcategory_summary["total_sales"] != 0, (subcategory_summary["total_profit"] / subcategory_summary["total_sales"]) * 100, np.nan)
top_subcategories = subcategory_summary.sort_values("total_sales", ascending=False).head(10)
lowest_profit_subcategories = subcategory_summary.sort_values("total_profit").head(10)

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=category_summary, x="Category", y="total_sales", ax=axes[0], palette="Set2")
axes[0].set_title("Category Sales")

sns.barplot(data=category_summary, x="Category", y="total_profit", ax=axes[1], palette="Set1")
axes[1].set_title("Category Profit")

sns.barplot(data=top_subcategories, x="Sub-Category", y="total_sales", hue="Category", ax=axes[2])
axes[2].set_title("Top 10 Sub-Categories By Sales")
axes[2].tick_params(axis="x", rotation=60)

plt.tight_layout()
plt.show()

display(category_summary)
display(top_subcategories)
display(lowest_profit_subcategories)

## Revenue Drivers
Revenue performance usually reflects several interacting forces. This block looks at region, segment, discount, shipping delay, and line-item economics so we can identify which dimensions are associated with stronger revenue or weaker profitability.

In [ ]:
region_summary = df.groupby("Region", as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    order_lines=("Order ID", "size"),
).sort_values("total_sales", ascending=False)

segment_summary = df.groupby("Segment", as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    average_discount=("Discount", "mean"),
).sort_values("total_sales", ascending=False)

discount_summary = df.groupby("Discount Bucket", dropna=False, as_index=False).agg(
    total_sales=("Sales", "sum"),
    total_profit=("Profit", "sum"),
    avg_profit_margin=("Profit Margin Pct", "mean"),
    row_count=("Order ID", "size"),
).sort_values("total_sales", ascending=False)

driver_corr = df[["Sales", "Profit", "Quantity", "Discount", "Ship Delay Days", "Profit Margin Pct"]].corr(numeric_only=True)
sample_df = df.sample(min(len(df), 5000), random_state=42).copy()

fig, axes = plt.subplots(1, 3, figsize=(22, 5))
sns.barplot(data=region_summary, x="Region", y="total_sales", ax=axes[0], palette="rocket")
axes[0].set_title("Regional Revenue Contribution")

sns.barplot(data=discount_summary, x="Discount Bucket", y="total_profit", ax=axes[1], palette="coolwarm")
axes[1].set_title("Profit By Discount Bucket")
axes[1].tick_params(axis="x", rotation=30)

sns.scatterplot(data=sample_df, x="Discount", y="Profit", hue="Category", alpha=0.35, ax=axes[2])
axes[2].set_title("Discount vs Profit")

plt.tight_layout()
plt.show()

display(region_summary)
display(segment_summary)
display(discount_summary)
display(driver_corr)

## Moving Averages And Trend Signals
Moving averages smooth short-term volatility and make it easier to judge whether the business is improving, flattening, or weakening. This block compares recent daily sales with short and long moving averages and highlights the weakest monthly periods observed in the dataset.

In [ ]:
recent_window = daily_trend.dropna(subset=["sales_ma_30", "sales_ma_90"]).copy()
latest_signal = recent_window.iloc[-1].to_dict() if not recent_window.empty else {}
trend_signal = "insufficient_history"
if latest_signal:
    if latest_signal["sales_ma_30"] > latest_signal["sales_ma_90"]:
        trend_signal = "short_term_above_long_term"
    elif latest_signal["sales_ma_30"] < latest_signal["sales_ma_90"]:
        trend_signal = "short_term_below_long_term"
    else:
        trend_signal = "short_term_equal_long_term"

below_ma_periods = daily_trend[daily_trend["daily_sales"] < daily_trend["sales_ma_30"]].copy()
weak_period_summary = pd.Series({
    "trend_signal": trend_signal,
    "days_below_30d_ma": int(len(below_ma_periods)),
    "lowest_month_sales": float(monthly_trend["monthly_sales"].min()),
    "lowest_month_profit": float(monthly_trend["monthly_profit"].min()),
    "lowest_month_id": monthly_trend.loc[monthly_trend["monthly_sales"].idxmin(), "YearMonth"],
    "lowest_profit_month_id": monthly_trend.loc[monthly_trend["monthly_profit"].idxmin(), "YearMonth"],
}).to_frame(name="value")

display(weak_period_summary)

## Actionable Business Insights
The next cell turns the computed tables into concise business-facing takeaways. The notebook does not hardcode findings in advance; it derives them from the actual results produced above.

In [ ]:
top_category_sales = category_summary.iloc[0]
top_category_profit = category_summary.sort_values("total_profit", ascending=False).iloc[0]
weakest_sales_month = monthly_trend.loc[monthly_trend["monthly_sales"].idxmin()]
weakest_profit_month = monthly_trend.loc[monthly_trend["monthly_profit"].idxmin()]
top_region = region_summary.iloc[0]
worst_discount_bucket = discount_summary.sort_values("total_profit").iloc[0]
top_subcategory = top_subcategories.iloc[0]

insights = [
    f"Top revenue category: {top_category_sales['Category']} with sales of {top_category_sales['total_sales']:.2f}.",
    f"Top profit category: {top_category_profit['Category']} with profit of {top_category_profit['total_profit']:.2f}.",
    f"Top revenue sub-category: {top_subcategory['Sub-Category']} with sales of {top_subcategory['total_sales']:.2f}.",
    f"Weakest month by sales: {weakest_sales_month['YearMonth']} with sales of {weakest_sales_month['monthly_sales']:.2f}.",
    f"Weakest month by profit: {weakest_profit_month['YearMonth']} with profit of {weakest_profit_month['monthly_profit']:.2f}.",
    f"Highest revenue region: {top_region['Region']} with sales of {top_region['total_sales']:.2f}.",
    f"Lowest-profit discount bucket: {worst_discount_bucket['Discount Bucket']} with total profit of {worst_discount_bucket['total_profit']:.2f}.",
    f"Current moving-average signal: {trend_signal.replace('_', ' ')}.",
]

for item in insights:
    print(f"- {item}")

## Limitations
- This is a descriptive analysis of one sample retail dataset, not a causal experiment.
- Revenue and profit patterns can differ sharply, so sales growth should not be treated as automatically healthy.
- The dataset reflects one specific business context and time span; patterns may not transfer directly to other retailers.
- Some operational drivers such as marketing spend, stockouts, and competitor activity are not present in the workbook.

## Recommendations / Next Steps
- Investigate why the weakest profit months underperform even when sales are not the lowest.
- Review discount-heavy segments and sub-categories for margin leakage before expanding promotions.
- Use high-performing periods and categories as inputs for forecasting, assortment planning, and campaign timing.
- Extend this notebook with customer cohort analysis or region-level seasonality comparisons if deeper planning is needed.

## Common Mistakes
- Ranking product lines by revenue alone and ignoring profit or discount exposure.
- Treating monthly peaks as sustainable growth without checking moving averages.
- Calling a weak month seasonal after seeing only a single low observation.
- Ignoring the possibility that aggressive discounts raise sales while damaging profitability.

## Mini Challenge
1. Repeat the monthly trend analysis at the region level and identify whether weak periods are concentrated in one geography.
2. Build a category-by-quarter view and compare revenue winners with profit winners.
3. Test whether higher-discount buckets consistently deliver weaker average profit margins across all categories.

## Final Summary / Key Takeaways
This notebook builds a complete retail sales trend analysis workflow from transactional data to business insights. It covers long-run sales direction, seasonality, category and sub-category performance, revenue and margin drivers, moving averages, and weak-period detection. The main discipline is simple: validate the data first, compare revenue against profit second, and only then turn patterns into decisions.